# RSNA 2022 PED: Drive zip -> GCS

这个笔记本用于：
- 挂载 Google Drive
- 认证 GCS
- 将 Drive 中的 zip 解压后上传到 GCS 指定目录

你只需要修改第一个代码单元的超参即可复用。

In [ ]:
# Cell 1: 超参配置（后续修改只改这里）
PROJECT_ID = "brats-preprocess"
GCS_BUCKET = "rsna_2022_ped_preprocessed_npy"

# 显式超参：zip 文件名 与 GCS 目标文件夹名
ZIP_FILE_NAME = "npy1.zip"
TARGET_GCS_FOLDER = "npy1"

# 你的 Drive 内 zip 所在路径
DRIVE_ZIP_PATH = f"/content/drive/MyDrive/dataset/pretrain/RSNA-2022-PED/{ZIP_FILE_NAME}"

# 上传目标前缀
GCS_TARGET_PREFIX = f"gs://{GCS_BUCKET}/{TARGET_GCS_FOLDER}"

print("Configuration:")
print(f"- PROJECT_ID: {PROJECT_ID}")
print(f"- GCS_BUCKET: {GCS_BUCKET}")
print(f"- ZIP_FILE_NAME: {ZIP_FILE_NAME}")
print(f"- TARGET_GCS_FOLDER: {TARGET_GCS_FOLDER}")
print(f"- DRIVE_ZIP_PATH: {DRIVE_ZIP_PATH}")
print(f"- GCS_TARGET_PREFIX: {GCS_TARGET_PREFIX}")

In [ ]:
# Cell 2: 挂载 Drive + 认证 GCS
from google.colab import drive, auth

drive.mount('/content/drive')
auth.authenticate_user()

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

!gcloud config set project {PROJECT_ID}
!gcloud config get-value project
!gsutil ls -L -b gs://{GCS_BUCKET}

In [ ]:
# Cell 3: 边解压边上传到 GCS（不落盘整包）
import os
import zipfile
import shutil
import subprocess

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"Zip not found: {DRIVE_ZIP_PATH}")

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    file_infos = [info for info in zf.infolist() if not info.is_dir()]

    print(f"Zip file: {DRIVE_ZIP_PATH}")
    print(f"Total files to upload: {len(file_infos)}")

    # 预览前几个条目
    for info in file_infos[:10]:
        print(" -", info.filename)

    uploaded = 0
    for idx, info in enumerate(file_infos, start=1):
        rel_path = info.filename.lstrip('/').replace('\\', '/')
        gcs_uri = f"{GCS_TARGET_PREFIX}/{rel_path}"

        with zf.open(info, 'r') as src:
            proc = subprocess.Popen(
                ['gsutil', 'cp', '-', gcs_uri],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            shutil.copyfileobj(src, proc.stdin, length=1024 * 1024)
            proc.stdin.close()
            stdout, stderr = proc.communicate()

            if proc.returncode != 0:
                raise RuntimeError(
                    f"Upload failed for {rel_path}\n"
                    f"stdout: {stdout.decode('utf-8', errors='ignore')}\n"
                    f"stderr: {stderr.decode('utf-8', errors='ignore')}"
                )

        uploaded += 1
        if idx <= 5 or idx % 200 == 0 or idx == len(file_infos):
            print(f"Uploaded {idx}/{len(file_infos)} -> {gcs_uri}")

print(f"Streaming upload finished. Uploaded files: {uploaded}")

In [ ]:
# Cell 4: 上传结果检查
print("Sample objects under target prefix:")
!gsutil ls {GCS_TARGET_PREFIX}/ | head -20

print("\nSample .npy objects:")
!gsutil ls -r {GCS_TARGET_PREFIX}/**/*.npy | head -20

In [ ]:
# Cell 5: 统计 npy 文件数量（可选）
!gsutil ls -r {GCS_TARGET_PREFIX}/**/*.npy | wc -l